<a href="https://colab.research.google.com/github/roalddalhwriter/isro/blob/main/3DElevation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rasterio plotly -q
print('✅ Done')

✅ Done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DTM_PATH   = '/content/drive/MyDrive/Elevation/DTEEC_007671_2065_007882_2065_U01.IMG'
ORTHO_PATH = '/content/drive/MyDrive/Elevation/PSP_007882_2065_RED_A_01_ORTHO.JP2'

print('✅ Done')

Mounted at /content/drive
✅ Done


In [ ]:
import rasterio
import numpy as np

with rasterio.open(DTM_PATH) as src:
    dtm_raw       = src.read(1).astype(np.float32)
    dtm_transform = src.transform
    dtm_nodata    = src.nodata
    scale         = src.scales[0]  if src.scales  else 1.0
    offset        = src.offsets[0] if src.offsets else 0.0

dtm = dtm_raw * scale + offset
if dtm_nodata is not None:
    dtm[dtm_raw == dtm_nodata] = np.nan

DTM_PX_M = abs(dtm_transform.a)
print(f'DTM shape     : {dtm.shape}')
print(f'DTM px scale  : {DTM_PX_M:.3f} m/px')
print(f'Elevation     : {np.nanmin(dtm):.1f} m → {np.nanmax(dtm):.1f} m')
print('✅ DTM loaded')

DTM shape     : (12139, 7351)
DTM px scale  : 1.007 m/px
Elevation     : -3231.6 m → -3169.3 m
✅ DTM loaded


In [ ]:
# ── SET THESE to your cone's DTM pixel coordinates ───────────────────
# Get these from your results CSV (cx_dtm, cy_dtm columns)
CONE_CX = 4926   # replace with actual cx_dtm
CONE_CY = 7912   # replace with actual cy_dtm
HALF    = 200    # half-window in DTM pixels (~200m each side at 1m/px)
Z_EXAGGERATION = 5   # vertical exaggeration factor, increase if cone looks flat
# ─────────────────────────────────────────────────────────────────────

r0 = max(0, CONE_CY - HALF); r1 = min(dtm.shape[0], CONE_CY + HALF)
c0 = max(0, CONE_CX - HALF); c1 = min(dtm.shape[1], CONE_CX + HALF)
dtm_chip = dtm[r0:r1, c0:c1]

print(f'Chip shape    : {dtm_chip.shape}')
print(f'Elevation     : {np.nanmin(dtm_chip):.1f} → {np.nanmax(dtm_chip):.1f} m')
print(f'Vertical span : {np.nanmax(dtm_chip) - np.nanmin(dtm_chip):.1f} m')
print('✅ DTM chip cropped')

Chip shape    : (400, 400)
Elevation     : -3223.7 → -3194.6 m
Vertical span : 29.1 m
✅ DTM chip cropped


In [ ]:
from rasterio.windows import Window
from rasterio.transform import rowcol
from rasterio.enums import Resampling

# DTM chip corners → geographic coords
geo_tl = dtm_transform * (c0, r0)
geo_br = dtm_transform * (c1, r1)

with rasterio.open(ORTHO_PATH) as ortho:
    # Geographic → ortho pixel
    or0, oc0 = rowcol(ortho.transform, geo_tl[0], geo_tl[1])
    or1, oc1 = rowcol(ortho.transform, geo_br[0], geo_br[1])

    or0 = int(max(0, min(or0, or1)))
    or1 = int(max(or0, min(max(or0, or1), ortho.height)))
    oc0 = int(max(0, min(oc0, oc1)))
    oc1 = int(max(oc0, min(max(oc0, oc1), ortho.width)))

    window = Window(oc0, or0, oc1 - oc0, or1 - or0)

    # Read downsampled to match DTM chip size exactly
    ortho_chip = ortho.read(
        1, window=window,
        out_shape=(dtm_chip.shape[0], dtm_chip.shape[1]),
        resampling=Resampling.bilinear
    ).astype(np.float32)

# Normalize to 0-255
p2, p98 = np.percentile(ortho_chip[ortho_chip > 0], (2, 98))
ortho_norm = np.clip(
    (ortho_chip - p2) / (p98 - p2 + 1e-6) * 255, 0, 255
).astype(np.uint8)

print(f'Ortho chip shape: {ortho_norm.shape}')
print(f'Pixel range     : {ortho_norm.min()} → {ortho_norm.max()}')
print('✅ Ortho chip ready')

Ortho chip shape: (400, 400)
Pixel range     : 0 → 255
✅ Ortho chip ready


In [ ]:
import plotly.graph_objects as go

# Build metric coordinate grids
rows, cols = dtm_chip.shape
x = np.arange(cols) * DTM_PX_M   # metres east
y = np.arange(rows) * DTM_PX_M   # metres north

# Fill NaN for smooth surface rendering
dtm_filled = dtm_chip.copy()
dtm_filled[np.isnan(dtm_filled)] = np.nanmean(dtm_chip)

# Apply vertical exaggeration
dtm_exag = dtm_filled * Z_EXAGGERATION

fig = go.Figure(data=[
    go.Surface(
        z            = dtm_exag,
        x            = x,
        y            = y,
        surfacecolor = ortho_norm,
        colorscale   = 'gray',
        showscale    = False,
        lighting     = dict(
            ambient   = 0.6,
            diffuse   = 0.8,
            specular  = 0.2,
            roughness = 0.5,
        ),
    )
])

# Actual elevation range for z axis labels (not exaggerated)
z_min = float(np.nanmin(dtm_chip))
z_max = float(np.nanmax(dtm_chip))

fig.update_layout(
    title = (f'3D Cone View — DTM ({CONE_CX}, {CONE_CY}) | '
             f'{Z_EXAGGERATION}x vertical exaggeration'),
    scene = dict(
        xaxis = dict(title='East (m)'),
        yaxis = dict(title='North (m)'),
        zaxis = dict(
            title      = 'Elevation (m)',
            tickvals   = [z_min * Z_EXAGGERATION,
                          ((z_min + z_max) / 2) * Z_EXAGGERATION,
                          z_max * Z_EXAGGERATION],
            ticktext   = [f'{z_min:.0f}',
                          f'{(z_min+z_max)/2:.0f}',
                          f'{z_max:.0f}'],
        ),
        aspectmode  = 'manual',
        aspectratio = dict(x=1, y=1, z=0.3),
        camera      = dict(
            eye = dict(x=1.5, y=-1.5, z=1.2)
        )
    ),
    width  = 950,
    height = 750,
    margin = dict(l=0, r=0, b=0, t=50)
)

fig.show()
print('✅ 3D plot rendered — use mouse to rotate, scroll to zoom')

✅ 3D plot rendered — use mouse to rotate, scroll to zoom


In [ ]:
OUTPUT_HTML = '/content/drive/MyDrive/cones/results_v6/cone_3d_view.html'
fig.write_html(OUTPUT_HTML)
print(f'✅ Saved interactive 3D view to {OUTPUT_HTML}')
print('Open the HTML file in any browser to view and rotate')

✅ Saved interactive 3D view to /content/drive/MyDrive/cones/results_v6/cone_3d_view.html
Open the HTML file in any browser to view and rotate
